# 07 — Compare previous and horizon-specific adapters fairly

This notebook evaluates the previous shared-horizon Qwen3-4B adapters and the new 1Q, 2Q, and 4Q adapters on exactly the same untouched test rows. It uses each adapter's original prompt format, reports accuracy by table and forecast scope, and produces a direct current-versus-previous decision for every horizon.


## 1. Mount the Colab project

Connect Google Drive and point the notebook at the same persistent project used by notebook 06.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.chdir("/content/drive/MyDrive/JobAI")
os.environ["JOBAI_REPO"] = "/content/drive/MyDrive/JobAI"

## 2. Install the evaluation stack

Install the same pinned model libraries used for fine-tuning. The base model itself is not downloaded again.

In [ ]:
%pip install --quiet -r requirements-train-colab.txt

## 3. Discover current and previous comparable adapter runs

The notebook requires the latest current-dataset `enhanced_v1` run for `(1,)`, `(2,)`, and `(4,)`. It also discovers the latest saved previous shared adapter for every feature-set and training-size design. Previous adapters may have an older dataset-card checksum, but they are re-evaluated here on the current untouched rows, so their old MAE is never compared directly with the new MAE.


In [ ]:
import hashlib, importlib.metadata, json, os, platform, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt

def find_repo():
    configured = os.environ.get("JOBAI_REPO")
    if configured:
        return Path(configured).resolve()
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "model.yaml").is_file():
            return candidate
    return current

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

REPO = find_repo()
PRO = REPO / "data" / "processed"
MAN = REPO / "data" / "manifests"
REPORTS = REPO / "reports"
FIGURES = REPORTS / "figures"
EVALUATION_ROOT = REPORTS / "model_evaluations"
FIGURES.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CFG = yaml.safe_load((REPO / "configs" / "model.yaml").read_text())
SEED = int(MODEL_CFG["training"]["seed"])
TEST_PATH = PRO / "panel_test.jsonl"
BASELINE_PATH = REPORTS / "baseline_predictions.csv"
BASELINE_MANIFEST_PATH = REPORTS / "baseline_run_manifest.json"
CARD_PATH = MAN / "panel_dataset_card.json"
RUN_MANIFEST_DIR = MAN / "finetune_runs"
for required in (TEST_PATH, BASELINE_PATH, BASELINE_MANIFEST_PATH, CARD_PATH):
    assert required.is_file(), f"Missing prerequisite: {required}"
assert RUN_MANIFEST_DIR.is_dir(), "Run notebook 06 for h1, h2, and h4 first"
panel_card = json.loads(CARD_PATH.read_text())
baseline_manifest = json.loads(BASELINE_MANIFEST_PATH.read_text())
assert panel_card["files"]["test"]["sha256"] == sha256(TEST_PATH), "Notebook 05 test checksum is stale"
assert baseline_manifest["outputs"]["reports/baseline_predictions.csv"]["sha256"] == sha256(BASELINE_PATH), "Notebook 04 predictions are stale"
assert baseline_manifest["selection_catalog_sha256"] == panel_card["selection_catalog_sha256"], "Notebooks 04 and 05 used different target catalogs"
current_card_sha = sha256(CARD_PATH)

available_runs = []
for manifest_path in sorted(RUN_MANIFEST_DIR.glob("*.json")):
    manifest = json.loads(manifest_path.read_text())
    adapter_dir = REPO / manifest.get("adapter_path", "")
    if manifest.get("base_model") != "Qwen/Qwen3-4B" or not adapter_dir.is_dir():
        continue
    training_cfg = manifest.get("model_config", {}).get("training", {})
    feature_set = manifest.get("feature_set", training_cfg.get("feature_set", "legacy_v1"))
    horizons = tuple(sorted(int(h) for h in manifest.get("training_horizons", training_cfg.get("training_horizons", []))))
    profile_name = manifest.get("profile_name", "shared_legacy")
    available_runs.append({**manifest, "manifest_path": manifest_path, "source_manifest_path": manifest_path, "adapter_dir": adapter_dir,
                           "feature_set": feature_set, "training_horizons": horizons, "profile_name": profile_name,
                           "train_count": int(manifest.get("train_examples_used", manifest.get("train_examples", -1))),
                           "max_seq_length": int(training_cfg.get("max_seq_length", 512)),
                           "panel_card_matches_current": manifest.get("panel_dataset_card_sha256") == current_card_sha})

# Older downloaded adapters may have lost their training manifests during a repository refresh.
# Recover their immutable metadata from Notebook 07 evaluation evidence instead of guessing from folder names.
known_run_ids = {run["run_id"] for run in available_runs}
recovered_previous_runs = 0
for evaluation_manifest_path in sorted(EVALUATION_ROOT.glob("*/evaluation_manifest.json")):
    evidence = json.loads(evaluation_manifest_path.read_text())
    run_id = evidence.get("run_id")
    if not run_id or run_id in known_run_ids or evidence.get("base_model") != "Qwen/Qwen3-4B":
        continue
    adapter_dir = REPO / evidence.get("adapter_path", "")
    horizons = tuple(sorted(int(h) for h in evidence.get("training_horizons", [])))
    if not adapter_dir.is_dir() or horizons != (1, 2, 4):
        continue
    feature_set = evidence.get("feature_set", "legacy_v1")
    available_runs.append({**evidence, "manifest_path": evaluation_manifest_path,
                           "source_manifest_path": evaluation_manifest_path, "adapter_dir": adapter_dir,
                           "profile_name": "recovered_previous_shared", "feature_set": feature_set,
                           "training_horizons": horizons, "train_count": int(evidence["train_examples"]),
                           "max_seq_length": 256 if feature_set == "legacy_v1" else 512,
                           "panel_card_matches_current": False})
    known_run_ids.add(run_id)
    recovered_previous_runs += 1
print("previous runs recovered from evaluation evidence:", recovered_previous_runs)

# The three current adapters must match the current Notebook 05 dataset exactly.
current_runs = []
for horizon in (1, 2, 4):
    matches = [run for run in available_runs if run["panel_card_matches_current"]
               and run["feature_set"] == "enhanced_v1" and run["training_horizons"] == (horizon,)]
    assert matches, f"Missing current-dataset enhanced Qwen3-4B h{horizon} adapter. Run Notebook 06 with that profile."
    run = sorted(matches, key=lambda item: item["run_id"])[-1]
    run["comparison_role"] = "current_horizon_specific"
    run["prompt_schema"] = "five_table_context_v5"
    current_runs.append(run)

# Keep the latest previous shared adapter for each feature-set/training-size design.
# Evaluating all retained designs prevents a weak historical run from being chosen as an easy opponent.
previous_by_design = {}
for run in available_runs:
    if run["training_horizons"] != (1, 2, 4):
        continue
    design = (run["feature_set"], run["train_count"])
    if design not in previous_by_design or run["run_id"] > previous_by_design[design]["run_id"]:
        previous_by_design[design] = run
previous_runs = sorted(previous_by_design.values(), key=lambda run: (run["feature_set"], run["train_count"]))
assert previous_runs, "No previous shared-horizon Qwen3-4B adapters were found for comparison"
for run in previous_runs:
    run["comparison_role"] = "previous_shared"
    run["prompt_schema"] = "legacy_v1" if run["feature_set"] == "legacy_v1" else "enhanced_shared_v1"
selected_runs = previous_runs + current_runs
assert all(not bool(run.get("test_data_used", run.get("test_data_used_for_training", False))) for run in selected_runs), (
    "A selected adapter manifest says test data was used during training"
)

MODEL_ID = "Qwen/Qwen3-4B"
candidate_by_id = {candidate["id"]: candidate for candidate in MODEL_CFG["candidates"]}
MODEL_CHOICE = candidate_by_id[MODEL_ID]
MODEL_ARCHITECTURE = MODEL_CHOICE.get("architecture", "causal_lm")
BASE_MODEL_DIR = REPO / "models" / "base" / MODEL_ID.replace("/", "--")
assert (BASE_MODEL_DIR / ".download_complete").is_file(), f"Persistent base model missing: {BASE_MODEL_DIR}"
assert torch.cuda.is_available(), "A CUDA GPU is required for model evaluation"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = bool(torch.cuda.is_bf16_supported())
for index, run in enumerate(selected_runs):
    horizon_slug = "-".join(map(str, run["training_horizons"]))
    feature_slug = re.sub(r"[^a-z0-9]+", "_", run["feature_set"].lower()).strip("_")
    role_slug = "new" if run["comparison_role"] == "current_horizon_specific" else "previous"
    run["model_label"] = f"qwen3_4b_{role_slug}_{feature_slug}_h{horizon_slug}_{run['train_count']}_examples_{index}"
    print(f"selected {run['comparison_role']}: {run['run_id']} | horizons={horizon_slug} | "
          f"examples={run['train_count']:,} | prompt={run['prompt_schema']} | current_panel={run['panel_card_matches_current']}")
print("GPU:", GPU_NAME, f"({GPU_VRAM_GB:.1f} GB)")


## 4. Build a large matched untouched-test sample

Each horizon receives up to 1,000 deterministic test cases for which all four baselines exist. This is large enough for a more stable decision than the previous 200-case comparison and keeps all model/baseline cases identical.


In [ ]:
TEST_PER_HORIZON = 1000
KEYS = ["table_id", "series_id", "horizon_q", "origin_quarter", "target_quarter"]
test_full = pd.read_json(TEST_PATH, lines=True)
assert (test_full["split"] == "test").all()
baseline_full = pd.read_csv(BASELINE_PATH)
baseline_test = baseline_full.loc[baseline_full["split"] == "test"].copy()
required_models = {"last_value", "seasonal_naive", "ridge", "ridge_enhanced"}
found_models = set(baseline_test["model"].unique())
assert required_models.issubset(found_models), f"Rerun Notebook 04; missing baselines: {sorted(required_models - found_models)}"
model_sets = baseline_test.groupby(KEYS)["model"].agg(set)
eligible_keys = model_sets[model_sets.map(lambda values: required_models.issubset(values))].reset_index()[KEYS]
eligible = test_full.merge(eligible_keys, on=KEYS, how="inner", validate="one_to_one")

def deterministic_group_sample(frame, n, seed):
    ranked = frame.copy()
    ranked["_sample_key"] = ranked["example_id"].map(lambda value: hashlib.sha256(f"{seed}|{value}".encode()).hexdigest())
    # Round-robin by table and scope prevents one large target family from dominating evaluation.
    ranked["_stratum"] = ranked[["table_id", "forecast_scope"]].astype(str).agg("|".join, axis=1)
    ranked["_round"] = ranked.groupby("_stratum")["_sample_key"].rank(method="first").astype(int)
    ranked["_stratum_key"] = ranked["_stratum"].map(lambda value: hashlib.sha256(f"{seed}|{value}".encode()).hexdigest())
    return ranked.sort_values(["_round", "_stratum_key", "_sample_key"]).head(min(n, len(ranked))).drop(
        columns=["_sample_key", "_stratum", "_round", "_stratum_key"]
    )

sample_parts = [deterministic_group_sample(group, TEST_PER_HORIZON, SEED + int(horizon))
                for horizon, group in eligible.groupby("horizon_q", sort=True)]
test_sample = pd.concat(sample_parts, ignore_index=True).sort_values(["horizon_q", "table_id", "series_id", "origin_quarter"]).reset_index(drop=True)
assert set(test_sample["horizon_q"]) == {1, 2, 4}
assert not test_sample.duplicated("example_id").any()
print("full untouched test examples:", len(test_full))
print("eligible matched examples   :", len(eligible))
display(test_sample.groupby(["horizon_q", "table_id", "forecast_scope"]).size().rename("selected").reset_index())


## 5. Load the base model once and attach every comparison adapter

Load the shared Qwen3-4B base model in four-bit mode only once. Attach the previous shared adapters and the new 1Q, 2Q, and 4Q LoRA adapters under unique names, avoiding repeated base-model downloads or loads.


In [ ]:
from transformers import AutoModelForCausalLM, AutoModelForMultimodalLM, AutoProcessor, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

compute_dtype = torch.bfloat16 if BF16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype,
)
first_adapter = selected_runs[0]["adapter_dir"]
if MODEL_ARCHITECTURE == "multimodal_text_only":
    processor = AutoProcessor.from_pretrained(first_adapter, local_files_only=True)
    tokenizer = processor.tokenizer
    model_class = AutoModelForMultimodalLM
else:
    processor = None
    tokenizer = AutoTokenizer.from_pretrained(first_adapter, local_files_only=True, use_fast=True)
    model_class = AutoModelForCausalLM
tokenizer.padding_side = "left"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = model_class.from_pretrained(
    BASE_MODEL_DIR, local_files_only=True, quantization_config=quantization_config,
    device_map="auto", dtype=compute_dtype,
)
first_name = selected_runs[0]["model_label"]
model = PeftModel.from_pretrained(base_model, first_adapter, adapter_name=first_name, local_files_only=True)
for run in selected_runs[1:]:
    model.load_adapter(run["adapter_dir"], adapter_name=run["model_label"], local_files_only=True)
model.eval()
print("Loaded one base model with adapters:", [run["model_label"] for run in selected_runs])


## 6. Generate matched forecasts from previous and new adapters

Every adapter is scored on the same deterministic rows for each horizon it supports. Previous adapters receive their earlier legacy or enhanced-shared prompt; new horizon adapters receive the current five-table context prompt. This avoids unfairly testing an old adapter with a prompt format it never learned. Parse failures remain visible and are never replaced with baseline predictions.


In [ ]:
CURRENT_SYSTEM_TEXT = ("You forecast Finnish job-vacancy time series from Statistics Finland and KEHA. "
                       "Return exactly one JSON object with one numeric field named target_scaled_change.")
PREVIOUS_SYSTEM_TEXT = ("You forecast Finnish registered job vacancies. "
                        "Return exactly one JSON object with one numeric field named target_scaled_change.")
GENERATION_BATCH_SIZE = 16

def format_number(value):
    return format(float(value), ".6g")

FEATURE_ALIASES = [
    ("quarter_of_year", "quarter"), ("qoq_change_scaled", "qoq"),
    ("yoy_change_scaled", "yoy"), ("recent_mean_4_scaled", "mean4"),
    ("window_mean_scaled", "mean_window"), ("recent_slope_4_scaled", "slope4"),
    ("window_slope_scaled", "slope_window"), ("recent_std_4_scaled", "std4"),
    ("window_std_scaled", "std_window"), ("zero_fraction_window", "zero_fraction"),
]

def compact_feature_text(engineered):
    return "; ".join(f"{alias}={format_number(engineered[key])}" for key, alias in FEATURE_ALIASES)

def slope(values):
    values = np.asarray(values, dtype=float)
    return float(np.polyfit(np.arange(len(values), dtype=float), values, 1)[0]) if len(values) > 1 else 0.0

def shift_quarter(quarter, offset):
    ordinal = int(quarter[:4]) * 4 + int(quarter[-1]) - 1 + int(offset)
    return f"{ordinal // 4}Q{ordinal % 4 + 1}"

def run_input_view(row, run):
    full_history = json.loads(row["input_values_json"])
    if run["comparison_role"] == "current_horizon_specific" or run["feature_set"] != "legacy_v1":
        return full_history, float(row["scale"]), json.loads(row["engineered_features_json"]), row["window_start_quarter"]
    # Legacy adapters predate the move from 8 to 12 history quarters; enhanced shared adapters use 12.
    history = [float(value) for value in full_history[-8:]]
    assert len(history) == 8, "Previous-adapter comparison requires at least eight history quarters"
    values = np.asarray(history, dtype=float)
    scale = max(abs(float(values[-1])), float(np.std(values)), 1.0)
    recent = values[-4:]
    engineered = {
        "quarter_of_year": int(str(row["origin_quarter"])[-1]),
        "qoq_change_scaled": float((values[-1] - values[-2]) / scale),
        "yoy_change_scaled": float((values[-1] - values[-5]) / scale),
        "recent_mean_4_scaled": float((np.mean(recent) - values[-1]) / scale),
        "window_mean_scaled": float((np.mean(values) - values[-1]) / scale),
        "recent_slope_4_scaled": float(slope(recent) / scale),
        "window_slope_scaled": float(slope(values) / scale),
        "recent_std_4_scaled": float(np.std(recent) / scale),
        "window_std_scaled": float(np.std(values) / scale),
        "zero_fraction_window": float(np.mean(values == 0)),
    }
    return history, scale, engineered, shift_quarter(row["origin_quarter"], -7)

def prompt_messages(row, run):
    history, prediction_scale, engineered, window_start = run_input_view(row, run)
    dimensions = json.loads(row["dimensions_json"])
    if run["prompt_schema"] == "five_table_context_v5":
        context_raw = row.get("hierarchical_context_json", "{}")
        context = json.loads(context_raw) if isinstance(context_raw, str) and context_raw else {}
        lines = [
            f"Source family: {row['series_family']}", f"Table: {row['table_id']}",
            f"Forecast scope: {row['forecast_scope']}", f"Measure: {row['measure_code']}",
            f"Dimensions: {json.dumps(dimensions, ensure_ascii=False, sort_keys=True)}",
            f"{len(history)} quarterly values, oldest to newest: {[format_number(v) for v in history]}",
            f"Origin-safe features: {compact_feature_text(engineered)}",
        ]
        if context:
            lines.append(f"Reference context: {json.dumps(context, ensure_ascii=False, sort_keys=True)}")
        lines.extend([f"Forecast origin: {row['origin_quarter']}",
                      f"Forecast horizon: {int(row['horizon_q'])} quarter(s)",
                      f"Scale: {format_number(prediction_scale)}",
                      "Predict target_scaled_change = (target vacancy value - latest history value) / scale."])
        return [{"role": "system", "content": CURRENT_SYSTEM_TEXT},
                {"role": "user", "content": chr(10).join(lines)}]
    feature_text = ""
    if run["prompt_schema"] == "enhanced_shared_v1":
        feature_text = f"Origin-safe features: {compact_feature_text(engineered)}\n"
    history_description = ("Eight quarterly vacancy values"
                           if run["prompt_schema"] == "legacy_v1" and len(history) == 8
                           else f"{len(history)} quarterly vacancy values")
    user_text = (
        f"Series family: {row['series_family']}\nTable: {row['table_id']}\n"
        f"Dimensions: {json.dumps(dimensions, ensure_ascii=False, sort_keys=True)}\n"
        f"History start: {window_start}\n"
        f"{history_description}, oldest to newest: {[format_number(v) for v in history]}\n"
        f"{feature_text}Forecast origin: {row['origin_quarter']}\n"
        f"Forecast horizon: {int(row['horizon_q'])} quarter(s)\nTarget quarter: {row['target_quarter']}\n"
        f"Scale: {format_number(prediction_scale)}\n"
        "Predict target_scaled_change = (target vacancy value - latest history value) / scale."
    )
    return [{"role": "system", "content": PREVIOUS_SYSTEM_TEXT},
            {"role": "user", "content": user_text}]

def parse_scaled_change(text):
    match = re.search(r"\{[^{}]*\}", text)
    if not match:
        return None
    try:
        value = float(json.loads(match.group(0))["target_scaled_change"])
        return value if np.isfinite(value) else None
    except (KeyError, TypeError, ValueError, json.JSONDecodeError):
        return None

prediction_rows = []
generation_seconds_by_run = {}
all_records = test_sample.to_dict(orient="records")
for run in selected_runs:
    records = [row for row in all_records if int(row["horizon_q"]) in run["training_horizons"]]
    model_label = run["model_label"]
    model.set_adapter(model_label)
    run_max_length = int(run["max_seq_length"])
    prompt_probe = [tokenizer.apply_chat_template(prompt_messages(row, run), tokenize=False,
                    add_generation_prompt=True, enable_thinking=False) for row in records]
    prompt_lengths = [len(tokenizer(prompt, add_special_tokens=False)["input_ids"]) for prompt in prompt_probe]
    print(f"{model_label}: prompt tokens median/max={int(np.median(prompt_lengths))}/{max(prompt_lengths)}, limit={run_max_length}")
    assert max(prompt_lengths) <= run_max_length
    started_at = time.time()
    for start in range(0, len(records), GENERATION_BATCH_SIZE):
        batch = records[start:start + GENERATION_BATCH_SIZE]
        prompts = [tokenizer.apply_chat_template(prompt_messages(row, run), tokenize=False, add_generation_prompt=True, enable_thinking=False) for row in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True, max_length=run_max_length).to(model.device)
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=32, do_sample=False, use_cache=True)
        prompt_width = inputs["input_ids"].shape[1]
        responses = tokenizer.batch_decode(generated[:, prompt_width:], skip_special_tokens=True)
        for row, response in zip(batch, responses):
            predicted_change = parse_scaled_change(response.strip())
            parsed = predicted_change is not None
            _, prediction_scale, _, _ = run_input_view(row, run)
            predicted_value = max(0.0, float(row["last_value"]) + predicted_change * prediction_scale) if parsed else np.nan
            prediction_rows.append({**{key: row[key] for key in ["example_id", *KEYS, "series_family", "forecast_scope", "measure_code"]},
                                    "run_id": run["run_id"], "train_examples": run["train_count"], "feature_set": run["feature_set"],
                                    "comparison_role": run["comparison_role"], "prompt_schema": run["prompt_schema"],
                                    "training_horizons": "-".join(map(str, run["training_horizons"])),
                                    "model": model_label, "y_true": float(row["target_value"]), "prediction_scale": prediction_scale,
                                    "y_pred": predicted_value, "parsed": parsed, "response": response.strip()})
        print(f"{model_label}: generated {min(start + len(batch), len(records)):,}/{len(records):,}", end=chr(13))
    generation_seconds_by_run[run["run_id"]] = time.time() - started_at
    run_predictions = [row for row in prediction_rows if row["run_id"] == run["run_id"]]
    print()
    print(f"{model_label}: {generation_seconds_by_run[run['run_id']] / 60:.1f} minutes, parse success={np.mean([row['parsed'] for row in run_predictions]):.1%}")
model_predictions = pd.DataFrame(prediction_rows)


## 7. Calculate matched metrics by horizon and product scope

The overall table is useful, but it can hide a weak national model behind thousands of detailed KEHA cases. This section therefore reports the same MAE, RMSE, MASE, and sMAPE separately for each table and forecast scope.


In [ ]:
sample_keys = test_sample[KEYS].drop_duplicates()
matched_baselines = baseline_test.merge(sample_keys, on=KEYS, how="inner", validate="many_to_one")
expected_adapter_counts = {int(horizon): sum(int(horizon) in run["training_horizons"] for run in selected_runs)
                           for horizon in (1, 2, 4)}
parsed_key_counts = (model_predictions.loc[model_predictions["parsed"]]
                     .groupby(KEYS)["model"].nunique().rename("parsed_adapter_count").reset_index())
parsed_key_counts["expected_adapter_count"] = parsed_key_counts["horizon_q"].map(expected_adapter_counts)
common_keys = parsed_key_counts.loc[
    parsed_key_counts["parsed_adapter_count"] == parsed_key_counts["expected_adapter_count"], KEYS
].drop_duplicates()
assert set(common_keys["horizon_q"]) == {1, 2, 4}, "No common parseable test rows for one or more horizons"
matched_baselines = matched_baselines.merge(common_keys, on=KEYS, how="inner", validate="many_to_one")
scale_lookup = matched_baselines.drop_duplicates(KEYS)[KEYS + ["mase_scale"]]
model_scored = (model_predictions.merge(common_keys, on=KEYS, how="inner", validate="many_to_one")
                .merge(scale_lookup, on=KEYS, how="left", validate="many_to_one"))
assert model_scored["mase_scale"].notna().all()
print("common scored rows by horizon:", common_keys.groupby("horizon_q").size().to_dict())
model_scored["split"] = "test"
model_scored["error"] = model_scored["y_true"] - model_scored["y_pred"]
model_scored["abs_error"] = model_scored["error"].abs()
model_scored["squared_error"] = model_scored["error"].pow(2)
model_scored["scaled_abs_error"] = model_scored["abs_error"] / model_scored["mase_scale"]
denom = (model_scored["y_true"].abs() + model_scored["y_pred"].abs()) / 2
model_scored["smape_component_pct"] = np.where(denom == 0, 0.0, 100 * model_scored["abs_error"] / denom)
metric_columns = ["model", "table_id", "series_family", "forecast_scope", "measure_code", "series_id", "horizon_q", "y_true", "y_pred", "abs_error", "squared_error", "scaled_abs_error", "smape_component_pct"]
combined = pd.concat([matched_baselines[metric_columns], model_scored.loc[model_scored["parsed"], metric_columns]], ignore_index=True)

def metric_summary(group):
    return pd.Series({"n": len(group), "MAE": group["abs_error"].mean(),
                      "RMSE": np.sqrt(group["squared_error"].mean()),
                      "MASE": group["scaled_abs_error"].mean(),
                      "sMAPE_pct": group["smape_component_pct"].mean()})

by_series = (combined.groupby(["model", "table_id", "series_family", "forecast_scope", "measure_code", "series_id", "horizon_q"], sort=True)
             .apply(metric_summary, include_groups=False).reset_index())
summary = (by_series.groupby(["model", "horizon_q"], sort=True)
           [["MAE", "RMSE", "MASE", "sMAPE_pct"]].mean().reset_index())
summary["n_series"] = by_series.groupby(["model", "horizon_q"])["series_id"].nunique().to_numpy()
parse_rates = model_predictions.groupby("model")["parsed"].mean().to_dict()
training_counts = model_predictions.groupby("model")["train_examples"].first().to_dict()
comparison_roles = model_predictions.groupby("model")["comparison_role"].first().to_dict()
prompt_schemas = model_predictions.groupby("model")["prompt_schema"].first().to_dict()
summary["parse_success"] = summary["model"].map(parse_rates).fillna(1.0)
summary["train_examples"] = summary["model"].map(training_counts)
summary["comparison_role"] = summary["model"].map(comparison_roles).fillna("baseline")
summary["prompt_schema"] = summary["model"].map(prompt_schemas).fillna("baseline")
display(summary.round(3))

by_scope = (by_series.groupby(["model", "table_id", "forecast_scope", "horizon_q"], sort=True)
            .agg(MAE=("MAE", "mean"), RMSE=("RMSE", "mean"), MASE=("MASE", "mean"),
                 sMAPE_pct=("sMAPE_pct", "mean"), n_series=("series_id", "nunique")).reset_index())
display(by_scope.round(3))


## 8. Compare new adapters with previous adapters and baselines

For every horizon, first compare the dedicated adapter with the strongest previous shared adapter on the identical test rows, then compare it with the strongest statistical baseline. Save both decisions and the exact test-sample fingerprint so results from different samples cannot be mistaken for a direct comparison.


In [ ]:
adapter_labels = [run["model_label"] for run in selected_runs]
best_baseline = (summary.loc[~summary["model"].isin(adapter_labels)]
                 .sort_values(["horizon_q", "MAE"]).groupby("horizon_q", as_index=False).first())
comparison_parts = []
for run in selected_runs:
    label = run["model_label"]
    adapter_summary = summary.loc[summary["model"] == label, ["horizon_q", "MAE", "RMSE", "MASE", "sMAPE_pct", "parse_success", "train_examples"]]
    part = adapter_summary.merge(best_baseline[["horizon_q", "model", "MAE"]], on="horizon_q", suffixes=("_adapter", "_best_baseline"))
    part["run_id"] = run["run_id"]
    part["adapter_model"] = label
    part["MAE_improvement_pct"] = 100 * (part["MAE_best_baseline"] - part["MAE_adapter"]) / part["MAE_best_baseline"]
    part["adapter_beats_best_baseline"] = part["MAE_adapter"] < part["MAE_best_baseline"]
    comparison_parts.append(part)
adapter_vs_baselines = pd.concat(comparison_parts, ignore_index=True)
adapter_only = summary.loc[summary["model"].isin(adapter_labels)].copy()
adapter_only["best_adapter_MAE"] = adapter_only.groupby("horizon_q")["MAE"].transform("min") == adapter_only["MAE"]
display(adapter_only.sort_values(["horizon_q", "MAE"]).round(3))
display(adapter_vs_baselines.round(3))

# This is the decisive apples-to-apples table: same rows, same targets, old versus new.
current_labels = [run["model_label"] for run in selected_runs if run["comparison_role"] == "current_horizon_specific"]
previous_labels = [run["model_label"] for run in selected_runs if run["comparison_role"] == "previous_shared"]
previous_best = (summary.loc[summary["model"].isin(previous_labels)]
                 .sort_values(["horizon_q", "MAE"]).groupby("horizon_q", as_index=False).first())
current_only = summary.loc[summary["model"].isin(current_labels)].copy()
paired_columns = ["horizon_q", "model", "MAE", "RMSE", "MASE", "sMAPE_pct", "n_series", "parse_success", "train_examples"]
current_vs_previous = current_only[paired_columns].merge(
    previous_best[paired_columns], on="horizon_q", suffixes=("_current", "_previous"), validate="one_to_one"
)
for metric in ("MAE", "RMSE", "MASE", "sMAPE_pct"):
    current_vs_previous[f"{metric}_improvement_pct"] = (
        100 * (current_vs_previous[f"{metric}_previous"] - current_vs_previous[f"{metric}_current"])
        / current_vs_previous[f"{metric}_previous"]
    )
current_vs_previous["current_beats_previous_MAE"] = current_vs_previous["MAE_current"] < current_vs_previous["MAE_previous"]
current_run_ids = {run["run_id"] for run in selected_runs if run["comparison_role"] == "current_horizon_specific"}
current_vs_baseline = adapter_vs_baselines.loc[adapter_vs_baselines["run_id"].isin(current_run_ids)]
replacement_passed = bool(
    len(current_vs_previous) == 3
    and current_vs_previous["current_beats_previous_MAE"].all()
    and current_vs_baseline["adapter_beats_best_baseline"].all()
    and (current_only["parse_success"] >= 0.99).all()
)
display(current_vs_previous.round(3))
print("replacement decision:", "PASS" if replacement_passed else "FAIL")

# Diagnose whether errors come from table family, target scale, zeros, or a few difficult series.
diagnostic_rows = pd.concat([
    matched_baselines.assign(parsed=True, run_id="baseline", train_examples=np.nan),
    model_scored,
], ignore_index=True, sort=False)
diagnostic_rows["target_size_band"] = pd.cut(
    diagnostic_rows["y_true"], bins=[-np.inf, 0, 25, 100, 500, np.inf],
    labels=["zero", "1-25", "26-100", "101-500", "over-500"]
)
diagnostic_rows["is_zero_target"] = diagnostic_rows["y_true"].eq(0)
error_diagnostics = (diagnostic_rows.groupby(
    ["model", "horizon_q", "table_id", "target_size_band"], observed=True, dropna=False
).agg(n=("y_true", "size"), n_series=("series_id", "nunique"),
      target_median=("y_true", "median"), median_AE=("abs_error", "median"),
      MAE=("abs_error", "mean"), RMSE=("squared_error", lambda values: float(np.sqrt(values.mean()))),
      MASE=("scaled_abs_error", "mean"), sMAPE_pct=("smape_component_pct", "mean"),
      parse_success=("parsed", "mean")).reset_index())
worst_series = (diagnostic_rows.groupby(["model", "horizon_q", "table_id", "series_id"], dropna=False)
                .agg(n=("y_true", "size"), target_median=("y_true", "median"),
                     median_AE=("abs_error", "median"), MAE=("abs_error", "mean"),
                     max_AE=("abs_error", "max")).reset_index()
                .sort_values(["model", "horizon_q", "MAE"], ascending=[True, True, False]))
display(error_diagnostics.round(3))

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

comparison_spec = {
    "selected_run_ids": [run["run_id"] for run in selected_runs],
    "selected_feature_sets": [run["feature_set"] for run in selected_runs],
    "selected_comparison_roles": [run["comparison_role"] for run in selected_runs],
    "selected_prompt_schemas": [run["prompt_schema"] for run in selected_runs],
    "selected_max_seq_lengths": [run["max_seq_length"] for run in selected_runs],
    "panel_test_sha256": sha256(TEST_PATH),
    "baseline_predictions_sha256": sha256(BASELINE_PATH),
    "test_examples_per_horizon": TEST_PER_HORIZON,
    "selected_example_ids_sha256": hashlib.sha256(chr(10).join(test_sample["example_id"].astype(str)).encode()).hexdigest(),
    "common_scored_keys_sha256": hashlib.sha256(common_keys.sort_values(KEYS).to_csv(index=False).encode()).hexdigest(),
}
comparison_id = hashlib.sha256(json.dumps(comparison_spec, sort_keys=True).encode()).hexdigest()[:12]
AGGREGATE_DIR = EVALUATION_ROOT / "comparisons" / f"comparison_{comparison_id}"
AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATE_PREDICTIONS_PATH = AGGREGATE_DIR / "adapter_test_predictions.csv"
AGGREGATE_SUMMARY_PATH = AGGREGATE_DIR / "adapter_and_baseline_metrics.csv"
ADAPTER_COMPARISON_PATH = AGGREGATE_DIR / "adapter_training_size_comparison.csv"
BASELINE_COMPARISON_PATH = AGGREGATE_DIR / "adapter_vs_baselines.csv"
CURRENT_VS_PREVIOUS_PATH = AGGREGATE_DIR / "current_vs_previous_adapters.csv"
ERROR_DIAGNOSTICS_PATH = AGGREGATE_DIR / "error_diagnostics.csv"
WORST_SERIES_PATH = AGGREGATE_DIR / "worst_series.csv"
SCOPE_METRICS_PATH = AGGREGATE_DIR / "metrics_by_scope.csv"
MATCHED_KEYS_PATH = AGGREGATE_DIR / "matched_test_keys.csv"
model_scored.to_csv(AGGREGATE_PREDICTIONS_PATH, index=False)
summary.to_csv(AGGREGATE_SUMMARY_PATH, index=False)
adapter_only.to_csv(ADAPTER_COMPARISON_PATH, index=False)
adapter_vs_baselines.to_csv(BASELINE_COMPARISON_PATH, index=False)
current_vs_previous.to_csv(CURRENT_VS_PREVIOUS_PATH, index=False)
error_diagnostics.to_csv(ERROR_DIAGNOSTICS_PATH, index=False)
worst_series.to_csv(WORST_SERIES_PATH, index=False)
by_scope.to_csv(SCOPE_METRICS_PATH, index=False)
common_keys.sort_values(KEYS).to_csv(MATCHED_KEYS_PATH, index=False)

comparison_manifest_path = AGGREGATE_DIR / "comparison_manifest.json"
comparison_manifest = {**comparison_spec, "comparison_id": comparison_id, "replacement_passed": replacement_passed,
                       "models": sorted(summary["model"].unique()),
                       "output_directory": str(AGGREGATE_DIR.relative_to(REPO))}
comparison_manifest_path.write_text(json.dumps(comparison_manifest, indent=2, ensure_ascii=False))
LATEST_COMPARISON_PATH = EVALUATION_ROOT / "latest_comparison_manifest.json"
LATEST_COMPARISON_PATH.write_text(json.dumps(comparison_manifest, indent=2, ensure_ascii=False))

for run in selected_runs:
    label = run["model_label"]
    run_dir = AGGREGATE_DIR / "runs" / run["run_id"]
    run_dir.mkdir(parents=True, exist_ok=True)
    prediction_path = run_dir / "test_predictions.csv"
    series_path = run_dir / "metrics_by_series.csv"
    summary_path = run_dir / "metrics_vs_baselines.csv"
    comparison_path = run_dir / "decision_vs_best_baseline.csv"
    run_prediction_rows = model_scored.loc[model_scored["run_id"] == run["run_id"]]
    run_series = by_series.loc[by_series["model"] == label]
    run_summary = pd.concat([summary.loc[summary["model"] == label], summary.loc[~summary["model"].isin(adapter_labels)]], ignore_index=True)
    run_comparison = adapter_vs_baselines.loc[adapter_vs_baselines["run_id"] == run["run_id"]]
    run_prediction_rows.to_csv(prediction_path, index=False)
    run_series.to_csv(series_path, index=False)
    run_summary.to_csv(summary_path, index=False)
    run_comparison.to_csv(comparison_path, index=False)
    parse_rate = float(run_prediction_rows["parsed"].mean())
    passed = bool(parse_rate >= 0.99 and run_comparison["adapter_beats_best_baseline"].all())
    evaluation_manifest = {
        "status": "pilot_passed" if passed else "pilot_failed",
        "run_id": run["run_id"], "source_manifest": str(run["source_manifest_path"].relative_to(REPO)),
        "base_model": MODEL_ID, "adapter_path": str(run["adapter_dir"].relative_to(REPO)),
        "train_examples": run["train_count"], "feature_set": run["feature_set"],
        "comparison_role": run["comparison_role"], "prompt_schema": run["prompt_schema"],
        "training_horizons": list(run["training_horizons"]),
        "test_examples_available": len(test_full), "test_examples_used": len(run_prediction_rows),
        "test_examples_per_horizon": TEST_PER_HORIZON, "test_data_used_for_training": False,
        "parse_success_rate": parse_rate, "minimum_parse_success_rate": 0.99,
        "adapter_beats_best_baseline_all_horizons": bool(run_comparison["adapter_beats_best_baseline"].all()),
        "generation_seconds": generation_seconds_by_run[run["run_id"]], "gpu": GPU_NAME,
        "outputs": {str(path.relative_to(REPO)): {"sha256": sha256(path)} for path in [prediction_path, series_path, summary_path, comparison_path]},
    }
    evaluation_manifest_path = run_dir / "evaluation_manifest.json"
    evaluation_manifest_path.write_text(json.dumps(evaluation_manifest, indent=2, ensure_ascii=False))
    print("saved run evidence:", run_dir)
print("new beats best previous by MAE:", current_vs_previous.set_index("horizon_q")["current_beats_previous_MAE"].to_dict())
print("saved aggregate comparison:", AGGREGATE_DIR)


## 9. Compare test MAE visually

The chart puts previous shared adapters, new horizon-specific adapters, and baselines on the same matched sample. Shorter bars are better. The decisive file is `current_vs_previous_adapters.csv`; use `metrics_by_scope.csv` to verify that an aggregate improvement is not hiding weak national or regional forecasts.


In [ ]:
%matplotlib inline
plot_data = summary.pivot(index="horizon_q", columns="model", values="MAE").sort_index()
ax = plot_data.plot(kind="bar", figsize=(13, 6), width=0.82)
ax.set(title="Matched untouched-test comparison: previous vs horizon-specific Qwen3-4B adapters", xlabel="Forecast horizon (quarters)", ylabel="Macro-average MAE")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
FIGURE_PATH = FIGURES / "qwen3_4b_previous_vs_horizon_adapters_matched_mae.png"
plt.savefig(FIGURE_PATH, dpi=160, bbox_inches="tight")
plt.show()
print("saved:", FIGURE_PATH)


## Decision rule

Do not call the new design better merely because its standalone MAE looks smaller or larger. It passes the replacement test only when `current_vs_previous_adapters.csv` shows `current_beats_previous_MAE = True` for H1, H2, and H4 on the matched rows, parse success is at least 99%, and each new adapter also beats the strongest matching baseline. Review `metrics_by_scope.csv` separately: aggregate improvement does not compensate for unusable national or regional forecasts.
